# **Noise2Void (3D), powered by CAREamics**

---

<font size = 4> Noise2Void is a deep-learning method that can be used to denoise many types of images, including microscopy images, originally published by [Krull *et al.* at CVPR 2019](https://openaccess.thecvf.com/content_CVPR_2019/papers/Krull_Noise2Void_-_Learning_Denoising_From_Single_Noisy_Images_CVPR_2019_paper.pdf). It denoises image data in a self-supervised manner, so high-quality, low-noise reference images are **not** required to train the network. A random subset of pixels is "masked" and the network learns to predict their values; the output is a denoised version of the image.

<font size = 4> **This notebook runs Noise2Void on 3D datasets using [CAREamics](https://careamics.github.io/), a modern PyTorch/Lightning implementation. Your images must be 3D stacks (`ZYX` by default; other axis orders can be set in section 3.1). For 2D datasets, use the **CAREamics** Noise2Void 2D notebook instead.**

---

<font size = 4>This notebook is based on:

<font size = 4>**Noise2Void - Learning Denoising from Single Noisy Images**, Krull *et al.*, CVPR 2019 (https://openaccess.thecvf.com/content_CVPR_2019/papers/Krull_Noise2Void_-_Learning_Denoising_From_Single_Noisy_Images_CVPR_2019_paper.pdf)

<font size = 4>**Please cite the original Noise2Void paper and CAREamics when using this notebook.**

# **How to use this notebook?**

---

<font size = 4>This notebook is structured in numbered sections. Run the cells from top to bottom.

---
### **Structure of a notebook**

<font size = 4>**Text cells** provide information. **Code cells** contain code; move your cursor over the `[ ]` on the left and click the play button to execute.

---
### **Making changes to the notebook**

<font size = 4>**Make a copy** of this notebook and before editing. To edit a cell, double-click on it.

# **0. Before getting started**
---

<font size = 4>Make sure your data is in a folder the notebook can access.

<font size = 4>Noise2Void trains on **noisy images alone**, with no clean reference and no second acquisition. A single noisy stack is enough, but several can be used. Only `.tif` and `.tiff` files are supported in this notebook.

<font size = 4>Every image must be a **3D stack** whose axes are ordered `ZYX` (slices, height, width) by default. If your stacks use a different axis order, or a file holds several stacks, set the `axes` field in section 3.1 to match. If your data is 2D, use the **CAREamics** Noise2Void 2D notebook instead.

<font size = 4>**We strongly recommend also preparing a Quality Control dataset**: a few noisy stacks with matching high signal-to-noise versions, using **matching file names** across the two folders. Section 5.2 uses these to measure how much the model improved your images.

<font size = 4>A common data structure that works well:

*   Data
    - **Training**: noisy 3D .tif stacks
    - **Quality control** (optional but recommended)
        - Low SNR stacks: img_1.tif, img_2.tif ...
        - High SNR stacks: img_1.tif, img_2.tif ...
    - **Prediction**: stacks to denoise
    - **Results**

---
<font size = 4>**Important note**

<font size = 4>To **train from scratch**: run **sections 1-4**, then **section 5** to assess quality and **section 6** to predict.
<font size = 4>To **continue training from a checkpoint**: run **sections 1-4** with `Use_pretrained_model` enabled and the training path filled in.
<font size = 4>To only **run predictions with an existing checkpoint**: run **sections 1-3**, enable `Use_pretrained_model`, run **section 4.1** to load the model, skip **section 4.2**, then run **section 6**.
---


# **1. Install CAREamics and dependencies**
---

## **1.1. Install CAREamics**

In [ ]:
#@markdown ##Install CAREamics and dependencies
#@markdown This installs a pinned, tested version of CAREamics.

!pip install "careamics==0.3.2" "careamics-portfolio" -q

print("CAREamics installed.")

## **1.2. Restart the runtime (only if you see an import error)**
<font size = 4>The install above leaves Colab's existing NumPy alone, so you can normally continue straight to section 1.3. **If section 1.3 raises a NumPy or import error**, go to `Runtime -> Restart session`, then re-run from section 1.3 (do **not** re-run the install cell).

## **1.3. Load key dependencies**

In [ ]:
#@markdown ##Load key dependencies
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile

import careamics
from careamics import CAREamist
from careamics.config import create_advanced_n2v_config

from careamics.metrics.metrics import scale_invariant_psnr
from microssim import micro_structural_similarity

TIFF_SUFFIXES = {".tif", ".tiff"}


def require_path(path_value, field_name):
    """Return `path_value` as a Path, with a clear error if the field was left unset.

    An empty form field becomes `Path(".")`, which exists, so CAREamics would silently
    search the whole working directory instead of reporting the mistake.
    Paths that must already exist are checked by CAREamics itself.
    """
    path_text = str(path_value).strip()
    if not path_text:
        raise ValueError(f"Please set `{field_name}` before running this cell.")
    return Path(path_text).expanduser()


def check_axes(axes, n_spatial):
    """Return `axes` upper-cased, with a clear error if it does not suit this notebook."""
    axes = str(axes).strip().upper()
    spatial = "ZYX" if n_spatial == 3 else "YX"
    if not axes or len(set(axes)) != len(axes) or any(a not in "STZYX" for a in axes):
        if "C" in axes:
            raise ValueError("Channels (C) are not supported in this notebook. "
                             "Save each channel as a separate file.")
        raise ValueError(f"Invalid axes '{axes}'. Use each letter at most once, "
                         "chosen from S, T, %s." % ", ".join(spatial))
    if any(a not in axes for a in spatial) or ("Z" in axes) != (n_spatial == 3):
        other = "2D" if n_spatial == 3 else "3D"
        raise ValueError(f"Axes '{axes}' must contain {', '.join(spatial)}"
                         f"{' and no Z' if n_spatial == 2 else ''} for this notebook; "
                         f"for {other} data use the {other} notebook.")
    return axes


def to_stack(array, axes, n_spatial):
    """Reorder an image read from a file to CAREamics' order (S, T, Z, Y, X) and fold
    the leading axes into one, giving an array of shape (n, *spatial)."""
    if array.ndim != len(axes):
        raise ValueError(f"The image has {array.ndim} dimensions {array.shape}, but the "
                         f"axes are '{axes}'. Set `axes` in section 3.1 to match your data.")
    order = sorted(range(len(axes)), key=lambda i: "STCZYX".index(axes[i]))
    array = np.transpose(array, order)
    return array.reshape(-1, *array.shape[-n_spatial:])


def fold_prediction(prediction, n_spatial):
    """Give a CAREamics prediction the same (n, *spatial) layout as `to_stack`."""
    prediction = np.asarray(prediction)
    return prediction.reshape(-1, *prediction.shape[-n_spatial:])


print(f"CAREamics version: {careamics.__version__}")

# **2. Initialise the Colab session**
---

## **2.1. Check for GPU access**

In [ ]:
#@markdown ##Run this cell to check if you have GPU access
import torch

if torch.cuda.is_available():
    print("You have GPU access.")
    print(torch.cuda.get_device_name(0))
else:
    print("You do NOT have GPU access.")
    print("Go to 'Runtime -> Change runtime type' and select a GPU hardware accelerator,")
    print("then re-run the notebook. Expect slow performance on CPU.")

## **2.2. Mount your Google Drive**

In [ ]:
#@markdown ##Play the cell to connect your Google Drive to Colab
#@markdown * Follow the instructions.
#@markdown * Click on "Files" on the left. Refresh it, and your Google Drive appears as "drive".

from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
#@markdown ##Optional: use an example dataset instead of your own data
#@markdown Leave this unchecked to use your own data from Google Drive. If checked, the
#@markdown dataset is downloaded to the Colab session and its location is printed below;
#@markdown paste the folders you need into the fields in section 3.1.
Use_example_dataset = False #@param {type:"boolean"}
#@markdown Dataset name from the [CAREamics portfolio](https://github.com/CAREamics/careamics-portfolio):
example_dataset = "Flywing" #@param {type:"string"}

if Use_example_dataset:
    from careamics_portfolio import PortfolioManager

    portfolio = PortfolioManager().denoising
    if not hasattr(portfolio, example_dataset):
        raise ValueError(f"Unknown dataset. Available: {portfolio.list_datasets()}")

    example_data_path = Path("/content/example_data")
    getattr(portfolio, example_dataset).download(example_data_path)
    print(f"Downloaded to {example_data_path}. Open the Files panel on the left to see "
          f"its folder structure, then set the section 3.1 paths accordingly.")
    print("This lives in the Colab session, not your Drive, and is lost on disconnect.")


# **3. Select your parameters and paths**
---

## **3.1. Setting the main training parameters**
<font size = 4>`Training_source` should point to a folder of noisy 3D `.tif` or `.tiff` stacks, or to a single TIFF stack. Noise2Void needs nothing else: no clean reference and no second acquisition.

<font size = 4>`model_path` is where the trained model, its checkpoints and the quality control results are written. Point it at a folder where you want the outputs to be saved.

<font size = 4>**Axes.** The default `ZYX` is one 3D stack per file. If a file holds several stacks, add `S` (independent stacks) or `T` (time points), for example `SZYX` or `TZYX`; if your dimensions come in another order, write them in that order, for example `ZXY`. The same axes apply to every file you load, including the quality control and prediction folders. If you load a pre-trained model, this field is ignored.

<font size = 4>**Patch size** is set separately in `Z` and in `XY`, because 3D microscopy stacks are usually anisotropic, with far fewer slices than pixels within a slice. Each value must be a **power of 2 and at least 8** (8, 16, 32, 64 ...) and must not exceed the corresponding dimension of your stacks. If your stacks have very few slices, use `patch_size_z = 8`.

<font size = 4>**Batch size** is the number of patches seen per training step. 3D patches are much larger than 2D ones, so the default is far smaller than in the 2D notebook; lower it further if you run out of GPU memory.

<font size = 4>**Prediction tile size** is used in sections 5.2 and 6.1. Large stacks are denoised tile by tile to keep memory use low, and is again set separately in `Z` and `XY`. Tiles overlap by 8 slices in `Z` and 48 pixels in `XY`, so the tile size must be comfortably larger than that: keep `Z` at 16 or more and `XY` at 128 or more. Reduce them if you run out of GPU memory.

<font size = 4>**Reproducibility.** Training is random in several ways: which patches are drawn from your images, how each one is augmented and, for Noise2Void, which pixels are masked. By default CAREamics draws a new random seed every run, so the same data and settings give a slightly different model each time. Leave **`Use_fixed_seed`** enabled to pin the seed, which makes a run repeatable and is what you want when comparing settings. Change `seed` to see how much run-to-run variation your data produces, or disable `Use_fixed_seed` to let CAREamics pick one at random.

<font size = 4>Two caveats: on a GPU a few operations remain nondeterministic, so results can still differ in the last decimal places; and a pre-trained model loaded in section 3.3 keeps the seed stored in its own checkpoint.


In [ ]:
#@markdown ###Path to the noisy training stack(s) (folder of 3D .tif/.tiff stacks, or a single file):
Training_source = "" #@param {type:"string"}

#@markdown ###Axes
axes = "ZYX" #@param {type:"string"}

#@markdown ###Model name and output folder:
model_name = "my_n2v_3d_model" #@param {type:"string"}
model_path = "" #@param {type:"string"}

#@markdown ###Training parameters
#@markdown Number of epochs:
number_of_epochs = 100 #@param {type:"number"}
#@markdown Patch size within a slice (pixels, square, a power of 2 and at least 8):
patch_size_xy = 64 #@param {type:"number"}
#@markdown Patch size across slices (a power of 2 and at least 8):
patch_size_z = 16 #@param {type:"number"}
#@markdown Batch size:
batch_size = 8 #@param {type:"number"}
#@markdown Number of patches held out for validation:
n_val_patches = 8 #@param {type:"number"}

#@markdown ###Prediction parameters
#@markdown Tile size within a slice, used in sections 5.2 and 6.1 (pixels, square):
prediction_tile_size_xy = 128 #@param {type:"number"}
#@markdown Tile size across slices (at least 16):
prediction_tile_size_z = 32 #@param {type:"number"}

# CAREamics needs one overlap value per axis in 3D; its 2D default of (48, 48) does
# not apply here. These are the values used by the CAREamics 3D N2V example, and
# they set the minimum usable tile sizes quoted above.
prediction_tile_overlap = (8, 48, 48)
prediction_tile_size = (
    int(prediction_tile_size_z),
    int(prediction_tile_size_xy),
    int(prediction_tile_size_xy),
)

#@markdown ###Reproducibility
#@markdown Use a fixed random seed, so that the same data and settings reproduce the same result:
Use_fixed_seed = True #@param {type:"boolean"}
#@markdown Seed value:
seed = 42 #@param {type:"number"}


## **3.2. Data augmentation**
<font size = 4>Data augmentation (flips and 90° rotations) usually improves results and is recommended.

In [ ]:
#@markdown ##Enable or disable data augmentation:
Use_Data_augmentation = True #@param {type:"boolean"}

## **3.3. Using a pre-trained model**
<font size = 4>You can continue training from a previously trained CAREamics model. Provide the path to a checkpoint (`.ckpt`). The pre-trained model's configuration is reused, so the parameters above are ignored when this is enabled.

In [ ]:
#@markdown ##Load weights from a pre-trained CAREamics model
Use_pretrained_model = False #@param {type:"boolean"}
#@markdown ###If enabled, provide the path to the checkpoint (.ckpt) file:
pretrained_model_path = "" #@param {type:"string"}

# **4. Train the network**
---

## **4.1. Prepare the training data and model**

In [ ]:
#@markdown ##Create the configuration and the CAREamist
# Augmentations: None -> default (flips + 90-degree rotations); [] -> disabled
augmentations = None if Use_Data_augmentation else []

work_dir = require_path(model_path, "model_path")
work_dir.mkdir(parents=True, exist_ok=True)

if Use_pretrained_model:
    checkpoint_path = require_path(pretrained_model_path, "pretrained_model_path")
    print(f"Loading pre-trained model from: {checkpoint_path}")
    print("The `axes` field is ignored: the model keeps the axes it was trained with.")
    careamist = CAREamist(checkpoint_path=checkpoint_path, work_dir=work_dir)
else:
    axes = check_axes(axes, 3)
    config = create_advanced_n2v_config(
        experiment_name=model_name,
        data_type="tiff",
        axes=axes,
        patch_size=(patch_size_z, patch_size_xy, patch_size_xy),
        batch_size=batch_size,
        num_epochs=number_of_epochs,
        n_val_patches=n_val_patches,
        augmentations=augmentations,
        seed=int(seed) if Use_fixed_seed else None,
    )
    print(config)
    careamist = CAREamist(config, work_dir=work_dir)

# The axes the model was trained with (not the form field, if a pre-trained model was
# loaded). Used below to read images in the same layout as the predictions.
data_axes = careamist.config.data_config.axes


## **4.2. Start training**
<font size = 4>Training checkpoints are saved automatically to your output folder. If you loaded a pre-trained model and only want to run predictions, leave the training path empty and skip this section.

In [ ]:
#@markdown ##Start training
training_source_text = str(Training_source).strip()

if not training_source_text:
    if Use_pretrained_model:
        print("No training path provided. Keeping the loaded model for evaluation/prediction.")
    else:
        raise ValueError("Please set Training_source before training.")
else:
    careamist.train(train_data=require_path(Training_source, "Training_source"))
    print("Training complete.")


# **5. Evaluate your model**
---

## **5.1. Inspection of the loss function**
---
<font size = 4>**Training loss** is the error on the patches the network learned from. **Validation loss** is the same error on a small set of patches held back from training, so it shows how the network does on data it has not seen.

<font size = 4>The loss will not approach zero, and the total change may look small. Noise2Void masks a small fraction of pixels and asks the network to predict them from their neighbours. The signal can be predicted this way but the noise cannot, so the loss settles at roughly the noise level of your data. A curve that drops only slightly and then flattens is expected, not a failed run.

<font size = 4>These curves tell you whether training converged. Whether the model improved your images is answered in **section 5.2**.


In [ ]:
#@markdown ##Plot the training and validation loss vs. epoch
try:
    loss_dict = careamist.get_losses()
except Exception as error:
    raise RuntimeError(
        "Could not read training losses. This section is only available after "
        "training in the current work_dir."
    ) from error

if not loss_dict.get("train_loss"):
    raise RuntimeError("No training loss values were found in the CSV logs.")

plt.figure(figsize=(8, 5))
plt.plot(loss_dict["train_epoch"], loss_dict["train_loss"], label="Train loss")
if loss_dict.get("val_loss"):
    plt.plot(loss_dict["val_epoch"], loss_dict["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Training losses")
plt.show()


## **5.2. Error mapping and quality metrics estimation**
---
<font size = 4>This section scores the model on a **Quality Control dataset**: pairs of low-SNR stacks (`Source_QC_folder`) and matching high-SNR stacks (`Target_QC_folder`), with **the same file name** in each folder.

<font size = 4>These images should be **held out from training**. Noise2Void trains on the noisy images themselves, so scoring it on data it was trained on tells you how well it fits that data, not how well it *generalises*.

<font size = 4>**SI-PSNR** (scale-invariant peak signal-to-noise ratio): how closely an image matches the ground truth. Higher is better; roughly +3 means the error was halved.

<font size = 4>**MicroSSIM**: structural similarity adapted to microscopy. It asks whether the same *structures* are present rather than whether pixel values match. Runs from 0 to 1, where 1 is a perfect match. MicroSSIM is a 2D measure, so it is computed slice by slice and averaged over the stack, while SI-PSNR is computed over the whole volume at once.

<font size = 4>The figure shows one `Z` slice of the input, the prediction and the ground truth, with a zoomed crop of each underneath. The MicroSSIM error map is saved as a 3D stack, one map per slice. Per-image scores are written to a `Quality Control` folder next to your model, with `QC_metrics_<model_name>.csv`.


In [ ]:
#@markdown ##Provide the Quality Control folders (paired low-SNR and high-SNR 3D .tif/.tiff stacks)
Source_QC_folder = "" #@param {type:"string"}
Target_QC_folder = "" #@param {type:"string"}
#@markdown Size of the zoomed-in crop shown under the full images (pixels, square):
qc_crop_size = 256 #@param {type:"number"}
#@markdown Z slice to display (-1 shows the middle slice of the stack):
qc_z_slice = -1 #@param {type:"number"}

source_qc_path = require_path(Source_QC_folder, "Source_QC_folder")
target_qc_path = require_path(Target_QC_folder, "Target_QC_folder")

# ZeroCostDL4Mic convention: QC results live in a "Quality Control" folder next to the
# model they describe.
qc_dir = work_dir / model_name / "Quality Control"
qc_dir.mkdir(parents=True, exist_ok=True)

# Let CAREamics list the source files itself, so that each prediction stays paired with
# the file it came from.
qc_predictions, qc_sources = careamist.predict(
    pred_data=source_qc_path,
    tile_size=prediction_tile_size,
    tile_overlap=prediction_tile_overlap,
)

csv_path = qc_dir / f"QC_metrics_{model_name}.csv"
scores = {"si_psnr_pred": [], "si_psnr_input": [], "ssim_pred": [], "ssim_input": []}
first_image = None  # kept for the figure; the rest are not held in memory

with open(csv_path, "w", newline="") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow([
        "image",
        "Prediction v. GT SI-PSNR",
        "Input v. GT SI-PSNR",
        "Prediction v. GT MicroSSIM",
        "Input v. GT MicroSSIM",
    ])

    for prediction, source in zip(qc_predictions, qc_sources):
        source_file = Path(source)
        target_file = target_qc_path / source_file.name
        if not target_file.exists():
            raise FileNotFoundError(
                f"No ground truth named '{source_file.name}' in {target_qc_path}. "
                "Paired QC images must have matching file names."
            )

        # Each file becomes (n_stacks, Z, Y, X); a plain 3D file has n_stacks = 1.
        # Scores are computed stack by stack and averaged.
        gt = to_stack(tifffile.imread(target_file), data_axes, 3).astype(np.float32)
        noisy = to_stack(tifffile.imread(source_file), data_axes, 3).astype(np.float32)
        denoised = fold_prediction(prediction, 3).astype(np.float32)
        if not gt.shape == noisy.shape == denoised.shape:
            raise ValueError(f"'{source_file.name}': source, target and prediction shapes "
                             "differ. Paired QC images must have the same size and axes.")

        # SI-PSNR is computed over the whole volume. MicroSSIM is a 2D measure:
        # given a stack it returns one score per slice, which we average.
        si_psnr_pred = float(np.mean([scale_invariant_psnr(g, d) for g, d in zip(gt, denoised)]))
        si_psnr_input = float(np.mean([scale_invariant_psnr(g, n) for g, n in zip(gt, noisy)]))
        ssim_pred = float(np.mean([np.mean(micro_structural_similarity(g, d))
                                   for g, d in zip(gt, denoised)]))
        ssim_input = float(np.mean([np.mean(micro_structural_similarity(g, n))
                                    for g, n in zip(gt, noisy)]))

        writer.writerow([source_file.name, si_psnr_pred, si_psnr_input,
                         ssim_pred, ssim_input])
        scores["si_psnr_pred"].append(si_psnr_pred)
        scores["si_psnr_input"].append(si_psnr_input)
        scores["ssim_pred"].append(ssim_pred)
        scores["ssim_input"].append(ssim_input)

        # Per-pixel MicroSSIM error map, saved for inspection in Fiji or napari.
        # The components have to be requested slice by slice: on a whole stack
        # microssim returns a list of results rather than a single one.
        error_map = np.stack([
            np.stack([
                np.asarray(
                    micro_structural_similarity(
                        g[z], d[z], return_individual_components=True
                    ).SSIM
                )
                for z in range(g.shape[0])
            ])
            for g, d in zip(gt, denoised)
        ]).astype(np.float32)
        if len(gt) == 1:
            error_map = error_map[0]
        tifffile.imwrite(qc_dir / f"MicroSSIM_GTvsPrediction_{source_file.name}", error_map)

        if first_image is None:
            first_image = (noisy, denoised, gt, si_psnr_input, si_psnr_pred)

        print(f"{source_file.name}: SI-PSNR {si_psnr_input:.2f} -> {si_psnr_pred:.2f}, "
              f"MicroSSIM {ssim_input:.4f} -> {ssim_pred:.4f}  (input -> prediction)")

label = "Mean over QC images" if len(scores["si_psnr_pred"]) > 1 else "QC image"
print(f"\n{'':32}{'input':>10}{'prediction':>13}{'gain':>10}")
print(f"{label + ', SI-PSNR:':32}"
      f"{np.mean(scores['si_psnr_input']):>10.2f}{np.mean(scores['si_psnr_pred']):>13.2f}"
      f"{np.mean(scores['si_psnr_pred']) - np.mean(scores['si_psnr_input']):>+10.2f}")
print(f"{label + ', MicroSSIM:':32}"
      f"{np.mean(scores['ssim_input']):>10.4f}{np.mean(scores['ssim_pred']):>13.4f}"
      f"{np.mean(scores['ssim_pred']) - np.mean(scores['ssim_input']):>+10.4f}")
print(f"\nMetrics and error maps saved to: {qc_dir}")

# Display one slice of the first QC stack: full field of view on top, a zoomed-in
# crop underneath.
noisy, denoised, gt, si_psnr_input, si_psnr_pred = first_image
if len(gt) > 1:
    print(f"The first QC file holds {len(gt)} stacks; showing the first.")
noisy, denoised, gt = noisy[0], denoised[0], gt[0]
z = gt.shape[0] // 2 if int(qc_z_slice) < 0 else int(qc_z_slice)
if z >= gt.shape[0]:
    raise IndexError(
        f"qc_z_slice must be between 0 and {gt.shape[0] - 1} for this stack."
    )
noisy, denoised, gt = noisy[z], denoised[z], gt[z]
height, width = gt.shape[-2:]
half = min(int(qc_crop_size), height, width) // 2
rows = slice(height // 2 - half, height // 2 + half)
cols = slice(width // 2 - half, width // 2 + half)

panels = [
    (noisy, f"Noisy input\nSI-PSNR: {si_psnr_input:.2f}"),
    (denoised, f"Prediction\nSI-PSNR: {si_psnr_pred:.2f}"),
    (gt, "Ground truth"),
]
print(f"\nShowing Z slice {z}.")
fig, ax = plt.subplots(2, 3, figsize=(15, 10))
for column, (image, title) in enumerate(panels):
    ax[0, column].imshow(image, cmap="gray")
    ax[0, column].set_title(title)
    ax[1, column].imshow(image[rows, cols], cmap="gray")
for axis in ax.ravel():
    axis.axis("off")
plt.show()


# **6. Using the trained model**
---

## **6.1. Generate predictions from an unseen dataset**

In [ ]:
#@markdown ###Path to the data to denoise and the folder where results are saved:
Data_folder = "" #@param {type:"string"}
Result_folder = "" #@param {type:"string"}

data_path = require_path(Data_folder, "Data_folder")
result_dir = require_path(Result_folder, "Result_folder")
result_dir.mkdir(parents=True, exist_ok=True)

predictions, sources = careamist.predict(
    pred_data=data_path,
    tile_size=prediction_tile_size,
    tile_overlap=prediction_tile_overlap,
)

for prediction, source in zip(predictions, sources):
    out_path = result_dir / (Path(source).stem + "_denoised.tif")
    tifffile.imwrite(out_path, np.asarray(prediction).squeeze().astype(np.float32))
    print(f"Saved: {out_path}")


## **6.2. Assess the predicted output**

In [ ]:
#@markdown ##Display one slice of an input stack next to its denoised prediction
idx = 0 #@param {type:"number"}
#@markdown Image to display if a file holds several (S or T axis):
stack_index = 0 #@param {type:"number"}
#@markdown Z slice to display (-1 shows the middle slice of the stack):
z_slice = -1 #@param {type:"number"}
idx = int(idx)
stack_index = int(stack_index)

if not predictions:
    raise RuntimeError("No predictions are available. Run section 6.1 first.")
if idx < 0 or idx >= len(predictions):
    raise IndexError(f"idx must be between 0 and {len(predictions) - 1}.")

# `sources` comes from CAREamics itself, so the input always matches the prediction.
input_stack = to_stack(tifffile.imread(sources[idx]), data_axes, 3)
pred_stack = fold_prediction(predictions[idx], 3)
if stack_index < 0 or stack_index >= len(input_stack):
    raise IndexError(f"stack_index must be between 0 and {len(input_stack) - 1} for this file.")
input_img, pred_img = input_stack[stack_index], pred_stack[stack_index]
file_label = Path(sources[idx]).name + (f", image {stack_index}" if len(input_stack) > 1 else "")

z = input_img.shape[0] // 2 if int(z_slice) < 0 else int(z_slice)
if z >= input_img.shape[0]:
    raise IndexError(
        f"z_slice must be between 0 and {input_img.shape[0] - 1} for this stack."
    )
input_img, pred_img = input_img[z], pred_img[z]

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(input_img, cmap="gray")
ax[0].set_title("Input (noisy)")
ax[1].imshow(pred_img, cmap="gray")
ax[1].set_title("Prediction (denoised)")
for axis in ax:
    axis.axis("off")
fig.suptitle(f"{file_label}, Z slice {z}")
plt.show()


## **6.3. Download your predictions**
---

<font size = 4>**Store your data** and all its results elsewhere by downloading them from your Google Drive, and then clean up the original folder tree (dataset, results, trained model) if you plan to train or use another network. Please note that the notebook will otherwise overwrite all files which have the same name.

# **7. Version log**
---
<font size = 4>**v1.0 (CAREamics)**:
*   First release of the CAREamics-powered Noise2Void 3D notebook.
*   Built on CAREamics 0.3.2 (`create_advanced_n2v_config` + `CAREamist`) with configurable `axes` (default `ZYX`).
*   Patch and prediction tile sizes are set separately in `Z` and `XY`, and prediction passes an explicit per-axis tile overlap.
*   Section 5.2 reports scale-invariant PSNR over the volume and MicroSSIM averaged over slices, for the prediction and for the noisy input against the ground truth, saves MicroSSIM error map stacks, and writes `QC_metrics_<model_name>.csv`.
*   Optional fixed random seed (section 3.1) for reproducible training and prediction.


# **Thank you for using Noise2Void 3D!**